## Silver - fact_pagamento

### Transformações para a camada silver

####Importando bibliotecas

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col, when
from pyspark.sql.types import StringType
from datetime import datetime
import pandas as pd



### fact_pagamento
####Consultando dados da tabela bronze

In [0]:
df_pagamento = spark.read.table("dev_credit_risk.bronze.fact_pagamento_raw_s3")

#### Início das transformações
#### Retirando os espaços em branco das colunas de tipo string.

In [0]:
for field in df_pagamento.schema.fields:
    if isinstance(field.dataType, StringType):
        df_pagamento = df_pagamento.withColumn(field.name, trim(col(field.name)))

####Transformando colunas de data em tipo date e normalizando valores de data

In [0]:
from pyspark.sql.functions import col, try_to_date, coalesce

def corrigir_multiplas_colunas_data(df, colunas, formatos=None):
    """
    Corrige múltiplos formatos de data em várias colunas de um DataFrame
    usando try_to_date para evitar quebras de parsing no Spark moderno.
    """
    if formatos is None:
        # Coloquei o formato com barra primeiro para priorizar o seu cenário
        formatos = ["yyyy/MM/dd", "yyyy-MM-dd", "dd/MM/yyyy", "dd-MM-yyyy"]
    
    for nome_coluna in colunas:
        # Trocamos to_date por try_to_date
        tentativas = [try_to_date(col(nome_coluna), fmt) for fmt in formatos]
        df = df.withColumn(nome_coluna, coalesce(*tentativas))
        
    return df

# --- Aplicação ---
colunas_para_corrigir = ["dtVencimento", "dtPagamento", "dtCompetencia"]

df_pagamento = corrigir_multiplas_colunas_data(df_pagamento, colunas_para_corrigir)
display(df_pagamento)
#display(df_pagamento.limit(100))

####Coluna "valorPago" recebe o valor da coluna "valorParcela"

In [0]:
df_pagamento = df_pagamento.withColumn("valorPago", col("valorParcela"))
df_pagamento.display()
# display(df_pagamento.filter(col("contratoId") == 1))

#### Corrigir coluna valorPago

In [0]:

df_pagamento = (
    df_pagamento
    .withColumn(
        "valorPago",
        when(
            col("dtPagamento").isNull(),
            None
        ).otherwise(col("valorPago"))
    )
)

In [0]:
display(df_pagamento.filter(col("diasAtraso") == 999))

#### transformando coluna de valor em decimal

In [0]:
from pyspark.sql.functions import col, when, regexp_replace

def normalizar_coluna_para_numerico(df, nome_coluna):
    """
    Substitui os valores da coluna original, padroniza o ponto decimal
    e altera o tipo da coluna para numérico Decimal(18,2).
    """
    return df.withColumn(
        nome_coluna,
        when(col(nome_coluna).contains(","), regexp_replace(col(nome_coluna), ",", "."))
        .otherwise(col(nome_coluna))
        .cast("decimal(18,2)") # Garante o tipo numérico com 2 casas decimais fixas
    )
# --- Aplicação direta no seu DataFrame ---
df_pagamento = normalizar_coluna_para_numerico(df_pagamento, "valorPago")
df_pagamento = normalizar_coluna_para_numerico(df_pagamento, "valorParcela")
# Verifique que o tipo mudou de string para decimal(18,2)
df_pagamento.printSchema()
# Visualizar o resultado final
display(df_pagamento)

####Exibindo o schema, quantidade de linhas e summary

In [0]:
df_pagamento.printSchema()
print(f"Qtd de linhas = {df_pagamento.count()}")

###Padronizando os nomes das colunas

In [0]:
rename_map = {
    "idPagamento": "id_pagamento",
    "idParcela": "id_parcela",
    "contratoId": "id_contrato",
    "numeroParcela": "numero_parcela",
    "dtVencimento": "dt_vencimento",
    "dtPagamento": "dt_pagamento",
    "dtCompetencia": "dt_competencia",
    "valorParcela": "valor_parcela",
    "valorPago": "valor_pago",
    "diasAtraso": "dias_atraso"
}

####Renomeando as colunas

In [0]:
for old_name, new_name in rename_map.items():
    df_pagamento = df_pagamento.withColumnRenamed(old_name, new_name)

####Exibindo o dataframe pronto para ser inserido na tabela delta silver

In [0]:
display(df_pagamento)

#### Escolhendo as colunas

In [0]:
df_pagamento = df_pagamento.select(
    "id_pagamento",
    "id_parcela",
    "id_contrato",
    "numero_parcela",
    "dt_vencimento",
    "dt_pagamento",
    "dt_competencia",
    "valor_parcela",
    "valor_pago",
    "dias_atraso"
)

#### Equalizando a coluna valorPago. 
#### Join com valorPago da fact_parcela.Inserindo os dados na tabela silver fact_pagamento

In [0]:
from pyspark.sql.functions import col, when

# Silver
df_parcela = spark.table("dev_credit_risk.silver.fact_parcela")

# Corrige valor_pago utilizando o valor da parcela
df_pagamento = (
    df_pagamento.alias("pg")
    .join(
        df_parcela.alias("p"),
        on="id_parcela",
        how="inner"
    )
    .select(
        col("pg.id_pagamento"),
        col("pg.id_parcela"),
        col("pg.id_contrato"),
        col("pg.numero_parcela"),
        col("pg.dt_competencia"),
        col("pg.dt_vencimento"),
        col("pg.dt_pagamento"),
        col("p.valor_parcela"),

        when(
            col("pg.dt_pagamento").isNull(),
            None
        ).otherwise(
            col("p.valor_parcela")
        ).alias("valor_pago"),

        col("pg.dias_atraso")
    )
)


#### Gravar os dados na tabela silver fact_pagamento

In [0]:
(
    df_pagamento
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dev_credit_risk.silver.fact_pagamento")
)

####Consultando os dados com SQL no schema tabela delta silver fact_pagamento

In [0]:
%sql
select * from dev_credit_risk.silver.fact_pagamento

In [0]:
%sql
select * from dev_credit_risk.silver.fact_pagamento